# **[_Handling CSV Ingestion with the Rescued Data Column_](url)**

We will focus on ingesting CSV files into Delta Lake using the CTAS ( CREATE TABLE AS SELECT ) pattren with the read_files() method and exploring the rescued data column.

In [0]:
%sql
SELECT current_catalog() as catalog, current_schema() as schema;

In [0]:
%sql
USE `pysaprk_demo`.`rescued_demo`;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS `pysaprk_demo`.`rescued_demo`.`rescude_data`;

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS pysaprk_demo.rescued_demo.sample_data")

## **_[SQL METHOD](url)_**

In [0]:
%sql
SELECT *
FROM csv.`/Volumes/pysaprk_demo/rescued_demo/rescude_data/rescued_01_clean.csv`
WITH (header = 'true', inferSchema = 'true', sep = ',', rescueddatacolumn = '_rescued_data');

In [0]:
%sql
select *
from read_files(
  '/Volumes/pysaprk_demo/rescued_demo/rescude_data/rescued_01_clean.csv',
  format => 'csv',
  header => 'true',
  inferSchema => 'true',
  sep => ',') 
;


In [0]:
%sql
-- 1. Drop the table if it exists for reproducibility
DROP TABLE IF EXISTS tb_rescued_01_clean;

-- 2. Create the table
CREATE TABLE rescued_demo.rescued_01_clean
AS
SELECT 
  *,
  _metadata.file_name as source_file_name,
  _metadata.file_path as source_file_path,
  _metadata.file_modification_time as source_file_modification_time,
  current_timestamp() as ingestion_timestamp
FROM read_files(
  '/Volumes/pysaprk_demo/rescued_demo/rescude_data/rescued_01_clean.csv',
  format => 'csv',
  header => 'true',
  inferSchema => 'true',
  sep => ','
);

-- 3. Display the table
SELECT * FROM rescued_demo.rescued_01_clean;

In [0]:
%sql
select source_file_name, count(*)
from rescued_demo.rescued_01_clean
group by 1
;

#### Data Type Missmatch

In [0]:
%sql
INSERT INTO pysaprk_demo.rescued_demo.rescued_01_clean
SELECT 
  *,
  _metadata.file_name as source_file_name,
  _metadata.file_path as source_file_path,
  _metadata.file_modification_time as source_file_modification_time,
  current_timestamp() as ingestion_timestamp
FROM read_files(
  '/Volumes/pysaprk_demo/rescued_demo/rescude_data/rescued_02_type_mismatch.csv',
  format => 'csv',
  header => 'true',
  schema => '''
  user_id string,
  email string,
  name string,
  age int,
  gender string,
  registration_date date''',
  rescueddatacolumn => '_rescued_data', -- this is the column name for the rescued data
  sep => ','
);

In [0]:
%sql
-- View the table
SELECT * FROM pysaprk_demo.rescued_demo.rescued_01_clean;

#### Data File with extra column

In [0]:
%sql
INSERT INTO pysaprk_demo.rescued_demo.rescued_01_clean
SELECT 
  *,
  _metadata.file_name as source_file_name,
  _metadata.file_path as source_file_path,
  _metadata.file_modification_time as source_file_modification_time,
  current_timestamp() as ingestion_timestamp
FROM read_files(
  '/Volumes/pysaprk_demo/rescued_demo/rescude_data/rescued_03_extra_columns.csv',
  format => 'csv',
  header => 'true',
  schema => '''
  user_id string,
  email string,
  name string,
  age int,
  gender string,
  registration_date date''',
  rescueddatacolumn => '_rescued_data', -- this is the column name for the rescued data
  sep => ','
);

#### Data File With Missing Columns

In [0]:
%sql
INSERT INTO pysaprk_demo.rescued_demo.rescued_01_clean
SELECT 
  *,
  _metadata.file_name as source_file_name,
  _metadata.file_path as source_file_path,
  _metadata.file_modification_time as source_file_modification_time,
  current_timestamp() as ingestion_timestamp
FROM read_files(
  '/Volumes/pysaprk_demo/rescued_demo/rescude_data/rescued_03_extra_columns.csv',
  format => 'csv',
  header => 'true',
  schema => '''
  user_id string,
  email string,
  name string,
  age int,
  gender string,
  registration_date date''',
  rescueddatacolumn => '_rescued_data', -- this is the column name for the rescued data
  sep => ','
);

In [0]:
%sql
-- View the table data
SELECT * FROM pysaprk_demo.rescued_demo.rescued_01_clean

## **_[PYTHON METHOD](url)_**

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType

## 1. Drop the table if exists
spark.sql("DROP TABLE IF EXISTS pysaprk_demo.rescued_demo.rescued_02_clean_py")

## 2. Define explicit schema (matching the SQL method)
schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("email", StringType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("registration_date", DateType(), True)
    # StructField("_rescued_data", StringType(), True)  # Add rescued data column to schema
])

## 3. Read the CSV files with rescued data column
df = (
  spark
  .read
  .format("csv")
  .option("header", "true")
  .option("mode", "PERMISSIVE")
  .option("columnNameOfCorruptRecord", "_rescued_data")
  .schema(schema)
  .load("/Volumes/pysaprk_demo/rescued_demo/rescude_data/")
)

## 4. Create a table from dataframe
df.write.mode("overwrite").saveAsTable("pysaprk_demo.rescued_demo.rescued_02_clean_py")

## 5. Read the table and show records with rescued data
spark.sql("SELECT user_id, email, age, `_rescued_data` FROM pysaprk_demo.rescued_demo.rescued_02_clean_py WHERE `_rescued_data` IS NOT NULL").show(n=10, truncate=False)